In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window
# from pyspark import pipelines as dp


In [0]:
dbutils.widgets.text("catalog", "workspace", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver_valid"

In [0]:
df = spark.read.table(SILVER)

In [0]:
DIM_VEHICLE = f"{CATALOG}.{SCHEMA}.dim_vehicle"
DIM_ROUTE = f"{CATALOG}.{SCHEMA}.dim_route"
DIM_DESTINATION = f"{CATALOG}.{SCHEMA}.dim_destination"
DIM_TIME = f"{CATALOG}.{SCHEMA}.dim_time"
FACT_VEHICLE_STATUS = f"{CATALOG}.{SCHEMA}.fact_vehicle_status"

In [0]:
dim_vehicle = spark.read.table(DIM_VEHICLE)
dim_route = spark.read.table(DIM_ROUTE)
dim_destination = spark.read.table(DIM_DESTINATION)
dim_time = spark.read.table(DIM_TIME)

In [0]:
fact_vehicle_status = (
    df
    .withColumn("date", F.to_date("event_time_local"))
    .withColumn("hour", F.hour("event_time_local"))
    .join(dim_vehicle.select("vehicleCode", "vehicle_key"), on="vehicleCode", how="left")
    .join(dim_route.select("route_id", "route_key"),  F.col("routeId")== F.col("route_id"), how="left")
    .join(dim_destination.select("headsign", "destination_key"), on="headsign", how="left")
    .join(dim_time.select("date", "hour", "time_key"), on =["date", "hour"], how = "left")
    .select("vehicle_key", "route_key", "destination_key", "time_key", "vehicleId","tripId", "event_time_local", "speed", "delay", "delay_min", "lat", "lon", "gpsQuality", "gps_ok", "has_trip", "is_delayed", "is_stopped", "is_moving", "delay_bucket")  
)

In [0]:
(
    fact_vehicle_status
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FACT_VEHICLE_STATUS)
)